[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/pypath/blob/main/notebooks/module8/06-testing-advanced.ipynb)

# Module 8 Lesson 6 — Advanced Testing with pytest

**Module 8: Best Practices & Real-World Python** | Estimated time: 30 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Use pytest **fixtures** with all four scopes (function, class, module, session)
- Write `conftest.py` to share fixtures across test files
- Use **yield fixtures** for setup and teardown
- Apply `@pytest.mark.parametrize` with IDs and indirect parametrization
- Mock external services with `pytest-mock` and `unittest.mock`
- Freeze time in tests with `freezegun`
- Measure and interpret **test coverage** with `pytest-cov`
- Test async code with `pytest-asyncio`
- Write property-based tests with `hypothesis`

In [ ]:
# Install all testing tools
!pip install pytest pytest-cov pytest-mock pytest-asyncio freezegun faker hypothesis --quiet
print("Testing tools installed.")

import os
os.makedirs("/tmp/testing_demo", exist_ok=True)
os.chdir("/tmp/testing_demo")
print("Working directory:", os.getcwd())

## The Application Under Test

First, let's write the actual code we want to test — a simple user management service.

In [ ]:
%%writefile myapp.py
"""User management service for testing demos."""
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime
from typing import Protocol
import hashlib
import re


class PasswordHasher(Protocol):
    def hash(self, password: str) -> str: ...
    def verify(self, password: str, hashed: str) -> bool: ...


@dataclass
class User:
    id: int
    username: str
    email: str
    password_hash: str
    created_at: datetime = field(default_factory=datetime.utcnow)
    is_active: bool = True


class UserRepository:
    def __init__(self) -> None:
        self._users: dict[int, User] = {}
        self._next_id: int = 1

    def save(self, user: User) -> User:
        self._users[user.id] = user
        return user

    def find_by_id(self, user_id: int) -> User | None:
        return self._users.get(user_id)

    def find_by_email(self, email: str) -> User | None:
        return next((u for u in self._users.values() if u.email == email), None)

    def delete(self, user_id: int) -> bool:
        if user_id in self._users:
            del self._users[user_id]
            return True
        return False

    def count(self) -> int:
        return len(self._users)

    def next_id(self) -> int:
        _id = self._next_id
        self._next_id += 1
        return _id


class SimpleHasher:
    def hash(self, password: str) -> str:
        return hashlib.sha256(password.encode()).hexdigest()

    def verify(self, password: str, hashed: str) -> bool:
        return self.hash(password) == hashed


class UserService:
    def __init__(self, repo: UserRepository, hasher: PasswordHasher) -> None:
        self.repo = repo
        self.hasher = hasher

    def register(self, username: str, email: str, password: str) -> User:
        if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
            raise ValueError(f"Invalid email: {email!r}")
        if len(password) < 8:
            raise ValueError("Password must be at least 8 characters")
        if self.repo.find_by_email(email):
            raise ValueError(f"Email already registered: {email!r}")
        user = User(
            id=self.repo.next_id(),
            username=username,
            email=email,
            password_hash=self.hasher.hash(password),
        )
        return self.repo.save(user)

    def authenticate(self, email: str, password: str) -> User | None:
        user = self.repo.find_by_email(email)
        if user and self.hasher.verify(password, user.password_hash):
            return user
        return None

    def deactivate(self, user_id: int) -> bool:
        user = self.repo.find_by_id(user_id)
        if not user:
            return False
        user.is_active = False
        self.repo.save(user)
        return True

## conftest.py — Shared Fixtures

`conftest.py` is a special pytest file. Fixtures defined there are automatically available to all tests in the same directory and subdirectories — no import needed.

In [ ]:
%%writefile conftest.py
"""Shared pytest fixtures."""
from __future__ import annotations
import pytest
from myapp import UserRepository, UserService, SimpleHasher, User
from datetime import datetime


# ── FUNCTION SCOPE (default) — fresh instance per test ───────────────────────

@pytest.fixture
def repo() -> UserRepository:
    """Fresh UserRepository for each test."""
    return UserRepository()


@pytest.fixture
def hasher() -> SimpleHasher:
    return SimpleHasher()


@pytest.fixture
def service(repo: UserRepository, hasher: SimpleHasher) -> UserService:
    """UserService wired with fresh repo and hasher."""
    return UserService(repo, hasher)


# ── YIELD FIXTURE — setup and teardown ────────────────────────────────────────

@pytest.fixture
def populated_repo(repo: UserRepository, hasher: SimpleHasher) -> UserRepository:
    """A repo pre-loaded with two users. Demonstrates yield fixture cleanup."""
    users = [
        User(id=repo.next_id(), username="alice", email="alice@example.com",
             password_hash=hasher.hash("password123"),
             created_at=datetime(2024, 1, 1)),
        User(id=repo.next_id(), username="bob", email="bob@example.com",
             password_hash=hasher.hash("securepass"),
             created_at=datetime(2024, 2, 1)),
    ]
    for user in users:
        repo.save(user)

    yield repo  # test runs here

    # Teardown: clear all users after the test
    # (In real code this might close a DB connection or delete temp files)
    repo._users.clear()
    print("\n[teardown] populated_repo cleared")


# ── MODULE SCOPE — created once per test module ───────────────────────────────

@pytest.fixture(scope="module")
def shared_hasher() -> SimpleHasher:
    """Single hasher shared across all tests in a module."""
    print("\n[setup] Creating shared_hasher")
    return SimpleHasher()


# ── SESSION SCOPE — created once for the entire test session ──────────────────

@pytest.fixture(scope="session")
def test_config() -> dict:
    """Configuration data available for the entire test session."""
    return {
        "base_url": "http://testserver",
        "api_version": "v1",
        "timeout": 30,
    }

## Fixtures in Depth + Parametrize

Now let's write tests that use these fixtures and show `@pytest.mark.parametrize`.

In [ ]:
%%writefile test_user_service.py
"""Tests for UserService."""
from __future__ import annotations
import pytest
from myapp import UserService


# ── Basic fixture usage ───────────────────────────────────────────────────────

class TestUserRegistration:
    def test_register_creates_user(self, service: UserService) -> None:
        user = service.register("alice", "alice@example.com", "password123")
        assert user.id == 1
        assert user.username == "alice"
        assert user.email == "alice@example.com"
        assert user.is_active is True

    def test_register_hashes_password(self, service: UserService) -> None:
        user = service.register("bob", "bob@example.com", "mysecret1")
        assert user.password_hash != "mysecret1"  # never store plaintext!
        assert len(user.password_hash) == 64       # SHA-256 hex digest

    def test_register_duplicate_email_raises(self, service: UserService) -> None:
        service.register("alice", "alice@example.com", "password123")
        with pytest.raises(ValueError, match="already registered"):
            service.register("alice2", "alice@example.com", "password456")


# ── @pytest.mark.parametrize ──────────────────────────────────────────────────

@pytest.mark.parametrize(
    "email",
    [
        pytest.param("notanemail",      id="no-at-sign"),
        pytest.param("@nodomain.com",   id="empty-local-part"),
        pytest.param("user@",           id="empty-domain"),
        pytest.param("",                id="empty-string"),
        pytest.param("a b@domain.com",  id="space-in-email"),
    ]
)
def test_invalid_email_raises(service: UserService, email: str) -> None:
    with pytest.raises(ValueError, match="Invalid email"):
        service.register("user", email, "validpassword")


@pytest.mark.parametrize(
    "password, should_raise",
    [
        pytest.param("short",        True,  id="too-short-5-chars"),
        pytest.param("1234567",      True,  id="too-short-7-chars"),
        pytest.param("12345678",     False, id="exactly-8-chars-ok"),
        pytest.param("longpassword", False, id="long-password-ok"),
    ]
)
def test_password_length_validation(
    service: UserService, password: str, should_raise: bool
) -> None:
    if should_raise:
        with pytest.raises(ValueError, match="Password must be"):
            service.register("user", "user@example.com", password)
    else:
        user = service.register("user", "user@example.com", password)
        assert user is not None


# ── Populated repo fixture ────────────────────────────────────────────────────

def test_authenticate_success(populated_repo, hasher) -> None:
    svc = UserService(populated_repo, hasher)
    user = svc.authenticate("alice@example.com", "password123")
    assert user is not None
    assert user.username == "alice"


def test_authenticate_wrong_password(populated_repo, hasher) -> None:
    svc = UserService(populated_repo, hasher)
    result = svc.authenticate("alice@example.com", "wrongpassword")
    assert result is None


def test_deactivate_user(populated_repo, hasher) -> None:
    svc = UserService(populated_repo, hasher)
    success = svc.deactivate(1)
    assert success is True
    user = populated_repo.find_by_id(1)
    assert user.is_active is False

## Mocking External Services with pytest-mock

Mocking lets you test code in isolation by replacing real dependencies (HTTP APIs, databases, email services) with controllable fakes.

In [ ]:
%%writefile email_service.py
"""Email sending service — makes real HTTP calls in production."""
import requests


class EmailService:
    def __init__(self, api_key: str, base_url: str = "https://api.sendgrid.com"):
        self.api_key = api_key
        self.base_url = base_url

    def send_welcome_email(self, to_email: str, username: str) -> dict:
        """Send a welcome email via external API."""
        payload = {
            "to": to_email,
            "subject": f"Welcome, {username}!",
            "body": f"Thanks for joining, {username}. Your account is ready.",
        }
        response = requests.post(
            f"{self.base_url}/v3/mail/send",
            json=payload,
            headers={"Authorization": f"Bearer {self.api_key}"},
        )
        response.raise_for_status()
        return {"status": "sent", "to": to_email}

    def send_password_reset(self, to_email: str, reset_token: str) -> dict:
        """Send a password reset email."""
        payload = {
            "to": to_email,
            "subject": "Password Reset Request",
            "body": f"Your reset token is: {reset_token}",
        }
        response = requests.post(
            f"{self.base_url}/v3/mail/send",
            json=payload,
            headers={"Authorization": f"Bearer {self.api_key}"},
        )
        response.raise_for_status()
        return {"status": "sent", "to": to_email}

In [ ]:
%%writefile test_email_service.py
"""Tests for EmailService using mocking."""
from __future__ import annotations
import pytest
from unittest.mock import MagicMock, patch
from email_service import EmailService


# ── Using mocker.patch (pytest-mock style) ────────────────────────────────────

def test_send_welcome_email_success(mocker) -> None:
    """Test that send_welcome_email makes the correct API call."""
    # Create a mock response object
    mock_response = MagicMock()
    mock_response.status_code = 202
    mock_response.raise_for_status.return_value = None

    # Patch requests.post wherever email_service uses it
    mock_post = mocker.patch("email_service.requests.post", return_value=mock_response)

    service = EmailService(api_key="test-key")
    result = service.send_welcome_email("user@example.com", "Alice")

    # Verify the result
    assert result == {"status": "sent", "to": "user@example.com"}

    # Verify the mock was called with the right arguments
    mock_post.assert_called_once()
    call_kwargs = mock_post.call_args
    assert "alice" in call_kwargs.kwargs["json"]["subject"].lower() or \
           "alice" in call_kwargs.kwargs["json"]["body"]
    assert call_kwargs.kwargs["headers"]["Authorization"] == "Bearer test-key"


def test_send_welcome_email_api_failure(mocker) -> None:
    """Test that HTTP errors are propagated correctly."""
    import requests as req
    mock_response = MagicMock()
    mock_response.raise_for_status.side_effect = req.HTTPError("500 Server Error")
    mocker.patch("email_service.requests.post", return_value=mock_response)

    service = EmailService(api_key="test-key")
    with pytest.raises(req.HTTPError):
        service.send_welcome_email("user@example.com", "Alice")


def test_send_password_reset_includes_token(mocker) -> None:
    """Test that the reset token is included in the email body."""
    mock_response = MagicMock()
    mock_response.raise_for_status.return_value = None
    mock_post = mocker.patch("email_service.requests.post", return_value=mock_response)

    service = EmailService(api_key="test-key")
    service.send_password_reset("user@example.com", "RESET-TOKEN-XYZ")

    body = mock_post.call_args.kwargs["json"]["body"]
    assert "RESET-TOKEN-XYZ" in body


# ── Using patch as a context manager (standard library style) ─────────────────

def test_send_welcome_email_with_context_manager() -> None:
    mock_response = MagicMock()
    mock_response.raise_for_status.return_value = None

    with patch("email_service.requests.post", return_value=mock_response) as mock_post:
        service = EmailService(api_key="key123")
        result = service.send_welcome_email("x@x.com", "Bob")
        assert result["status"] == "sent"
        assert mock_post.call_count == 1

## freezegun — Controlling Time in Tests

Code that depends on `datetime.now()` is notoriously hard to test. `freezegun` solves this.

In [ ]:
%%writefile time_service.py
from datetime import datetime, timedelta


def is_business_hours() -> bool:
    """Return True if current time is between 09:00 and 17:00 on a weekday."""
    now = datetime.now()
    return now.weekday() < 5 and 9 <= now.hour < 17


def token_expiry(hours: int = 24) -> datetime:
    """Return the datetime when a token created now will expire."""
    return datetime.utcnow() + timedelta(hours=hours)


def days_until_event(event_date: datetime) -> int:
    """Return the number of whole days until event_date from today."""
    delta = event_date.date() - datetime.utcnow().date()
    return delta.days

In [ ]:
%%writefile test_time_service.py
from datetime import datetime
import pytest
from freezegun import freeze_time
from time_service import is_business_hours, token_expiry, days_until_event


@freeze_time("2024-03-15 14:30:00")  # Friday 2:30 PM
def test_is_business_hours_weekday_afternoon() -> None:
    assert is_business_hours() is True


@freeze_time("2024-03-16 14:30:00")  # Saturday 2:30 PM
def test_is_business_hours_weekend() -> None:
    assert is_business_hours() is False


@freeze_time("2024-03-15 08:59:59")  # Friday just before 9am
def test_is_business_hours_before_open() -> None:
    assert is_business_hours() is False


@freeze_time("2024-01-01 12:00:00")
def test_token_expiry_24h() -> None:
    expiry = token_expiry(hours=24)
    assert expiry == datetime(2024, 1, 2, 12, 0, 0)


@freeze_time("2024-01-01 00:00:00")
def test_days_until_event() -> None:
    event = datetime(2024, 1, 11)
    assert days_until_event(event) == 10


@freeze_time("2024-01-01 00:00:00")
def test_days_until_past_event() -> None:
    past_event = datetime(2023, 12, 25)
    assert days_until_event(past_event) < 0

## pytest-cov — Measuring Test Coverage

In [ ]:
# Run all tests with coverage
!python -m pytest test_user_service.py test_email_service.py test_time_service.py \
    --cov=. \
    --cov-report=term-missing \
    --cov-omit="conftest.py,test_*.py" \
    -v 2>&1 | head -80

In [ ]:
# Run only user service tests with detailed output
!python -m pytest test_user_service.py -v --tb=short 2>&1

## pytest-asyncio — Testing Async Code

In [ ]:
%%writefile async_service.py
import asyncio
from typing import AsyncIterator


async def fetch_user(user_id: int) -> dict:
    """Simulate an async database call."""
    await asyncio.sleep(0.01)  # simulate I/O
    if user_id == 0:
        raise ValueError("User ID must be positive")
    return {"id": user_id, "name": f"User{user_id}", "active": True}


async def fetch_all_users(user_ids: list[int]) -> list[dict]:
    """Fetch multiple users concurrently."""
    results = await asyncio.gather(*[fetch_user(uid) for uid in user_ids])
    return list(results)


async def stream_events(count: int) -> AsyncIterator[dict]:
    """Yield events one by one, simulating a stream."""
    for i in range(count):
        await asyncio.sleep(0)
        yield {"event_id": i, "type": "update"}

In [ ]:
%%writefile test_async_service.py
import pytest
import pytest_asyncio
from async_service import fetch_user, fetch_all_users, stream_events


@pytest.mark.asyncio
async def test_fetch_user_returns_correct_data() -> None:
    user = await fetch_user(42)
    assert user["id"] == 42
    assert user["name"] == "User42"
    assert user["active"] is True


@pytest.mark.asyncio
async def test_fetch_user_invalid_id_raises() -> None:
    with pytest.raises(ValueError, match="must be positive"):
        await fetch_user(0)


@pytest.mark.asyncio
async def test_fetch_all_users_concurrent() -> None:
    users = await fetch_all_users([1, 2, 3, 4, 5])
    assert len(users) == 5
    ids = [u["id"] for u in users]
    assert sorted(ids) == [1, 2, 3, 4, 5]


@pytest.mark.asyncio
async def test_stream_events() -> None:
    events = []
    async for event in stream_events(3):
        events.append(event)
    assert len(events) == 3
    assert events[0]["event_id"] == 0
    assert all(e["type"] == "update" for e in events)

In [ ]:
!python -m pytest test_async_service.py -v --asyncio-mode=auto 2>&1

## Hypothesis — Property-Based Testing

Instead of specific test cases, Hypothesis generates hundreds of random inputs and tries to find counter-examples to your assertions.

In [ ]:
%%writefile test_hypothesis.py
from hypothesis import given, settings, strategies as st
from hypothesis import assume
import pytest


def encode_rle(text: str) -> str:
    """Run-length encode a string: 'aaa' -> '3a', 'ab' -> '1a1b'."""
    if not text:
        return ""
    result = []
    count = 1
    for i in range(1, len(text)):
        if text[i] == text[i - 1]:
            count += 1
        else:
            result.append(f"{count}{text[i-1]}")
            count = 1
    result.append(f"{count}{text[-1]}")
    return "".join(result)


def decode_rle(encoded: str) -> str:
    """Decode a run-length encoded string."""
    if not encoded:
        return ""
    result = []
    i = 0
    while i < len(encoded):
        count_chars = []
        while i < len(encoded) and encoded[i].isdigit():
            count_chars.append(encoded[i])
            i += 1
        if i < len(encoded):
            result.append(encoded[i] * int("".join(count_chars)))
            i += 1
    return "".join(result)


# Property: encoding then decoding returns the original string
@given(st.text(alphabet=st.characters(whitelist_categories=("Ll", "Lu")), min_size=1, max_size=50))
def test_rle_roundtrip(text: str) -> None:
    """encode(decode(text)) == text for any text."""
    assert decode_rle(encode_rle(text)) == text


# Property: encoded output is never longer than 2x the input (for single chars)
@given(st.text(alphabet="ab", min_size=1, max_size=100))
def test_rle_compression_bound(text: str) -> None:
    encoded = encode_rle(text)
    # Each unique character contributes at most 1 digit + 1 char
    assert len(encoded) <= len(text) * 2


# Property: sorted list is always sorted
@given(st.lists(st.integers()))
def test_sorted_is_sorted(lst: list[int]) -> None:
    result = sorted(lst)
    for i in range(len(result) - 1):
        assert result[i] <= result[i + 1]

In [ ]:
!python -m pytest test_hypothesis.py -v --tb=short 2>&1

## Practice Exercises

**Exercise 1 — Fixture scopes**
Create a `conftest.py` for a shopping cart application. Define:
- A `function`-scoped fixture `empty_cart` that returns a new empty cart
- A `module`-scoped fixture `product_catalogue` that returns a dict of 5 products
- A `yield` fixture `cart_with_items` that adds 3 items before the test and prints a cleanup message after

Write at least 4 tests that use these fixtures in different combinations.

**Exercise 2 — Parametrize with IDs**
Write a function `is_palindrome(s: str) -> bool`. Then write a parametrized test covering:
- Normal palindromes (`"racecar"`, `"madam"`)
- Non-palindromes (`"hello"`, `"python"`)
- Edge cases: empty string, single character, mixed case (decide on your implementation)
Use meaningful `id=` labels for each test case.

**Exercise 3 — Mock a weather API**
Write a `WeatherService` class with a method `get_temperature(city: str) -> float` that calls `requests.get` against a weather API. Write tests using `mocker.patch` that:
1. Return a mocked 200 response with temperature data
2. Simulate a 404 response and verify the correct exception is raised
3. Simulate a network timeout (`requests.exceptions.Timeout`) and verify it is handled